# ♾️🚀 HydraMultiRocketPlus

This notebook demonstrates the use of HydraMultiRocketPlus, a hybrid deep learning model for time series classification implemented in the tsai library (https://timeseriesai.github.io/tsai/). The model combines multi-head convolutional feature extraction with Rocket-style transformations, offering high performance on both clean and complex multivariate datasets.

Three benchmark datasets from the UCR/UEA archive are used to evaluate model performance:
- **BasicMotions**
  - 4-class human activity dataset with clean motion capture signals
  - 6 channels, 100 time steps, well-separated patterns

- **NATOPS**
  - 6-class gesture classification from hand signals recorded via motion sensors
  - 24 channels, 51 time steps, subtle inter-class variation

- **RacketSports**
  - 4-class sport activity recognition based on upper body motion
  - 6 channels, variable-length sequences (30–100 time steps), noisy real-world signals

In [ ]:
# BasicMotions (using tsai)

#!pip install tsai --quiet

from tsai.all import *
from tsai.all import get_UCR_data
from sklearn.preprocessing import LabelEncoder


ds_name = 'BasicMotions'
X, y, splits = get_UCR_data(ds_name, return_split=False)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Shape: {X.shape}, Classes: {np.unique(y)}")

dls = get_ts_dls(X, y_encoded, splits=splits)

model = HydraMultiRocketPlus(c_in=X.shape[1], c_out=len(np.unique(y)), seq_len=X.shape[2])
learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn.fit_one_cycle(10, 1e-3)


acc = learn.validate()
print(f"Final accuracy on BasicMotions: {acc}")




Shape: (80, 6, 100), Classes: ['badminton' 'running' 'standing' 'walking']


epoch,train_loss,valid_loss,accuracy,time
0,1.386293,1.107691,0.600000,00:08
1,0.952339,0.859437,0.750000,00:06
2,0.629542,0.527484,0.825000,00:07
3,0.467360,0.231945,0.950000,00:06
4,0.370074,0.092109,0.975000,00:07
5,0.305238,0.018925,1.000000,00:05
6,0.258946,0.003683,1.000000,00:08
7,0.224243,0.001090,1.000000,00:05
8,0.197267,0.000407,1.000000,00:06
9,0.175699,0.000182,1.000000,00:05


Final accuracy on BasicMotions: [0.00018162079504691064, 1.0]


In [ ]:
# NATOPS (gesture recognition)


X, y, splits = get_UCR_data("NATOPS", return_split=False)
le = LabelEncoder()
y = le.fit_transform(y)

dls = get_ts_dls(X, y, splits=splits)

model = HydraMultiRocketPlus(c_in=X.shape[1], c_out=len(np.unique(y)), seq_len=X.shape[2])

learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn.fit_one_cycle(10, 1e-3)

loss, acc = learn.validate()
print(f"NATOPS accuracy: {acc:.2f}")



epoch,train_loss,valid_loss,accuracy,time
0,1.466696,1.280660,0.472222,00:12
1,1.007202,0.788820,0.655556,00:10
2,0.825816,0.933433,0.750000,00:13
3,0.796211,0.628824,0.766667,00:15
4,0.742080,0.497706,0.794444,00:11
5,0.645105,0.713728,0.772222,00:11
6,0.589553,0.837376,0.788889,00:11
7,0.539623,0.661966,0.822222,00:10
8,0.485636,0.521496,0.833333,00:10
9,0.433101,0.467476,0.838889,00:10


✅ NATOPS accuracy: 0.84


In [ ]:
# RacketSports (noisy multivariate)

X, y, splits = get_UCR_data("RacketSports", return_split=False)
le = LabelEncoder()
y = le.fit_transform(y)

dls = get_ts_dls(X, y, splits=splits)

model = HydraMultiRocketPlus(c_in=X.shape[1], c_out=len(np.unique(y)), seq_len=X.shape[2])

learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn.fit_one_cycle(10, 1e-3)

loss, acc = learn.validate()
print(f"RacketSports accuracy: {acc:.2f}")


epoch,train_loss,valid_loss,accuracy,time
0,1.220210,4.603130,0.269737,00:04
1,0.831445,1.826381,0.421053,00:04
2,0.604682,3.589156,0.427632,00:05
3,0.490516,1.701282,0.592105,00:04
4,0.385149,0.986395,0.677632,00:04
5,0.316960,0.946245,0.710526,00:07
6,0.278239,0.804136,0.769737,00:04
7,0.238514,0.730557,0.815789,00:05
8,0.207554,0.695145,0.828947,00:04
9,0.182832,0.653215,0.848684,00:05


✅ RacketSports accuracy: 0.85
